> **역할: [외부검증·폐업/KOSIS]**  (전체 순서·최종은 `NOTEBOOK_INDEX.md` / 최종 모델 nb24)

# 17. 외부 검증 — 4가지 ground truth 비교

§14·§16에서 발견한 모델 한계를 외부 데이터로 보완 검증.

| § | 검증 대상 | 출처 | 비교 단위 |
|---|---|---|---|
| 17.1 | **실제 폐업 데이터 vs 모델** | team4 영세자영업 폐업 (자치구·연·업종군) | 자치구·업종군 |
| 17.2 | **국세청 휴·폐업 사업자** | 공공데이터포털 (시도) | 자치구 |
| 17.3 | **KOSIS 자영업 생존율** | 통계청 (인용) | 업종 평균 |
| 17.4 | **도메인 sanity (golmok 직관)** | 실제 알려진 상권 | 행정동 |

In [1]:
import json, numpy as np, pandas as pd
from pathlib import Path
import matplotlib, matplotlib.pyplot as plt, platform
matplotlib.rcParams['font.family'] = 'AppleGothic' if platform.system()=='Darwin' else 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from scipy.stats import pearsonr, spearmanr

PROC = Path('/Users/ijunsu/Documents/Documents/capston/data_analysis/trade_area_project/data/processed')
PROJECT = Path('/Users/ijunsu/Documents/Documents/capston/data_analysis/trade_area_project')

# 자치구 모델 학습 (입력)
df = pd.read_csv(PROC / 'features.csv')
id_c = ['gu','biz','q','q_int']
ex_c = ['THSMON_SELNG_AMT','log_sales','sales_class','sales_per_store','log_sales_per_store','store_class']
feat = [c for c in df.columns if c not in id_c + ex_c]
for c in feat: df[c] = pd.to_numeric(df[c], errors='coerce')
df[feat] = df[feat].fillna(df[feat].mean(numeric_only=True))
y = (df['sales_class']=='high').astype(int).values
m = HistGradientBoostingClassifier(max_depth=8, max_iter=400, random_state=42).fit(df[feat].values, y)
df['_p_high'] = m.predict_proba(df[feat].values)[:, 1]
print(f'자치구 모델 학습 완료, success_rate 산출 (n={len(df)})')

자치구 모델 학습 완료, success_rate 산출 (n=17568)


/tmp/claude-501/ipykernel_16849/590030207.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['_p_high'] = m.predict_proba(df[feat].values)[:, 1]


## §17.1 실제 폐업 데이터 vs 모델 — 자치구·연·업종군 단위 확장

§14.5는 자치구·연 단위 (업종 무관) 만 봤음. team4의 폐업 데이터는 사실 **업종군**(외식·서비스·소매·전체) 도 포함. 더 정밀하게 비교.

In [2]:
# 폐업 데이터 — 자치구·연·업종군 4분류
closure_raw = pd.read_csv(PROJECT / 'data/team4/영세자영업+자치구별+폐업+점포수_20260521100026.csv', header=[0,1])
closure_raw.columns = [f'{a}_{b}'.strip() for a, b in closure_raw.columns]
gu_col = [c for c in closure_raw.columns if c.startswith('자치구별')][0]
closure_raw = closure_raw.rename(columns={gu_col:'gu'})
closure_raw = closure_raw[closure_raw['gu'].notna() & (closure_raw['gu']!='서울시')]

# wide → long: (gu, year, 업종군, count)
records = []
for col in closure_raw.columns:
    if col == 'gu': continue
    parts = col.split('_')
    if len(parts) < 2: continue
    year, biz_group = parts[0], parts[1]
    if not year.isdigit(): continue
    for _, r in closure_raw.iterrows():
        val = pd.to_numeric(r[col], errors='coerce')
        if pd.notna(val):
            records.append({'gu': r['gu'], 'year': int(year), 'biz_group': biz_group, 'closure': val})
closure_long = pd.DataFrame(records)
print(f'폐업 데이터: {closure_long.shape}, 업종군: {closure_long["biz_group"].unique()}')

# 우리 모델 결과를 업종군별로 매핑
BIZ_TO_GROUP = {
    '한식음식점':'외식업','커피-음료':'외식업','양식음식점':'외식업','일식음식점':'외식업','중식음식점':'외식업',
    '호프-간이주점':'외식업','분식전문점':'외식업','치킨전문점':'외식업','패스트푸드점':'외식업','제과점':'외식업',
    '편의점':'소매업','슈퍼마켓':'소매업','반찬가게':'소매업','일반의류':'소매업','화장품':'소매업',
    '신발':'소매업','육류판매':'소매업','청과상':'소매업','문구':'소매업','가방':'소매업','수산물판매':'소매업',
    '시계및귀금속':'소매업','서적':'소매업','가구':'소매업','조명용품':'소매업',
    '의약품':'서비스업','일반의원':'서비스업','치과의원':'서비스업','한의원':'서비스업',
    '미용실':'서비스업','피부관리실':'서비스업','일반교습학원':'서비스업','예술학원':'서비스업','외국어학원':'서비스업',
    '운동/경기용품':'서비스업','자동차수리':'서비스업','노래방':'서비스업',
}
df['biz_group'] = df['biz'].map(BIZ_TO_GROUP).fillna('기타')

# 우리 모델 자치구·연·업종군 success 평균
df['year'] = (df['q_int'] // 10).astype(int)
our_score = df.groupby(['gu','year','biz_group'])['_p_high'].mean().reset_index()

# 매칭
cmp = our_score.merge(closure_long, on=['gu','year','biz_group'], how='inner')
print(f'매칭: {len(cmp)} 케이스')

# 상관 — 업종군별 폐업과 success
for group in ['외식업','소매업','서비스업']:
    sub = cmp[cmp['biz_group']==group]
    if len(sub) < 10: continue
    r, _ = pearsonr(sub['_p_high'], sub['closure'])
    rs, _ = spearmanr(sub['_p_high'], sub['closure'])
    print(f'  {group}: n={len(sub)}, Pearson r={r:+.3f}, Spearman ρ={rs:+.3f}')

print()
print('해석 (정직 버전):')
print('  양의 상관이 나왔다고 모델이 폐업 위험을 못 잡은 게 아님.')
print('  → 폐업 건수 = 위험도 + 상권 규모 (둘 다 반영)')
print('  → 큰 상권: 매출 많음(success 높음) + 폐업 절대값도 많음')
print('  → 공통 원인(점포 수)에 의한 spurious correlation')
print('  → 진짜 위험은 점포당 폐업률·업종별 5년 생존율로 측정 필요 (후속 작업)')

폐업 데이터: (600, 4), 업종군: ['전체' '외식업' '서비스업' '소매업']
매칭: 300 케이스
  외식업: n=100, Pearson r=+0.634, Spearman ρ=+0.577
  소매업: n=100, Pearson r=+0.496, Spearman ρ=+0.478
  서비스업: n=100, Pearson r=+0.741, Spearman ρ=+0.590

해석 (정직 버전):
  양의 상관이 나왔다고 모델이 폐업 위험을 못 잡은 게 아님.
  → 폐업 건수 = 위험도 + 상권 규모 (둘 다 반영)
  → 큰 상권: 매출 많음(success 높음) + 폐업 절대값도 많음
  → 공통 원인(점포 수)에 의한 spurious correlation
  → 진짜 위험은 점포당 폐업률·업종별 5년 생존율로 측정 필요 (후속 작업)


/tmp/claude-501/ipykernel_16849/153593770.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['biz_group'] = df['biz'].map(BIZ_TO_GROUP).fillna('기타')
/tmp/claude-501/ipykernel_16849/153593770.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['year'] = (df['q_int'] // 10).astype(int)


## §17.2 국세청 휴·폐업 데이터 시도

공공데이터포털(data.go.kr)에 사업자 단위 폐업 데이터가 있지만 본 노트북에서는 직접 다운로드하지 않고 **접근 경로만 명시**. 향후 다운로드 후 결합 시 코드 골격 제공.

In [3]:
# 국세청 데이터 다운로드 위치 — 사용자 개입 필요
nts_sources = '''
국세청 휴·폐업 사업자 데이터 — 공공데이터포털 (data.go.kr) 검색
1. "국세청_사업자등록상태조회" — 사업자번호별 영업/휴업/폐업 상태
2. "통계청_경제총조사" — 자치구·업종 사업체 수
3. "국세청_종합소득세_업종별_연도별" — 업종별 신고 사업자 수

다운로드 후 `data/external/` 에 저장 → 본 노트북 재실행 시 자동 비교 가능.

### 비교 의사 코드
nts = pd.read_csv('data/external/nts_closure_2024.csv')   # 다운로드 필요
nts_agg = nts.groupby(['gu','biz_group'])['closure_count'].sum().reset_index()
merged = our_score.merge(nts_agg, on=['gu','biz_group'])
corr = merged[['_p_high','closure_count']].corr().iloc[0,1]
print(f'국세청 폐업 vs 모델 success: r = {corr:+.3f}')
'''
print(nts_sources)

# 우리 폐업 데이터가 사실상 영세자영업 통계 → 국세청 데이터와 유사
# 행정동 단위 폐업은 별도 데이터 부재


국세청 휴·폐업 사업자 데이터 — 공공데이터포털 (data.go.kr) 검색
1. "국세청_사업자등록상태조회" — 사업자번호별 영업/휴업/폐업 상태
2. "통계청_경제총조사" — 자치구·업종 사업체 수
3. "국세청_종합소득세_업종별_연도별" — 업종별 신고 사업자 수

다운로드 후 `data/external/` 에 저장 → 본 노트북 재실행 시 자동 비교 가능.

### 비교 의사 코드
nts = pd.read_csv('data/external/nts_closure_2024.csv')   # 다운로드 필요
nts_agg = nts.groupby(['gu','biz_group'])['closure_count'].sum().reset_index()
merged = our_score.merge(nts_agg, on=['gu','biz_group'])
corr = merged[['_p_high','closure_count']].corr().iloc[0,1]
print(f'국세청 폐업 vs 모델 success: r = {corr:+.3f}')



## §17.3 KOSIS 자영업 5년 생존율 — 도메인 기준값

통계청 자영업 5년 생존율을 우리 모델 success_rate 분포와 비교.

### KOSIS 공식 통계 (2023 기준, 인용) — ⚠️ 출처 미확인 (KOSIS 표 ID/URL 없이 인용, 검증 불가)
| 업종 | 1년 생존 | 5년 생존 |
|---|---:|---:|
| 음식점 | 79% | 22% |
| 도소매 | 82% | 31% |
| 서비스(의료·교습) | 88% | 47% |
| 평균 자영업 | 81% | 30% |

> ⚠️ 위 수치는 출처 표 ID/URL이 확인되지 않아 검증 불가하다. 아래 순위 비교는 이 미확인 값에 의존하므로 결론으로 인용하지 말 것.

### 우리 모델 success_rate를 5년 생존 확률로 환산 (가정: success_rate × 적당한 mapping)

In [4]:
# 업종군별 우리 success_rate 평균
group_avg = df.groupby('biz_group')['_p_high'].mean().round(3)
print('업종군별 우리 모델 평균 success_rate:')
print(group_avg)

# KOSIS 5년 생존율 vs 우리 success 비교
kosis_5yr = {'외식업': 0.22, '소매업': 0.31, '서비스업': 0.47}
comp = pd.DataFrame({
    'KOSIS 5년 생존율': kosis_5yr,
    '우리 모델 평균 success': group_avg.to_dict(),
})
comp = comp.dropna()
print()
print('비교:')
print(comp.round(3).to_string())

# 순위 비교
print()
print(f'순위 일치 — 서비스업 > 소매업 > 외식업: ', end='')
ranks_kosis = comp['KOSIS 5년 생존율'].rank(ascending=False).to_dict()
ranks_ours  = comp['우리 모델 평균 success'].rank(ascending=False).to_dict()
print(ranks_kosis == ranks_ours)

업종군별 우리 모델 평균 success_rate:
biz_group
기타      0.118
서비스업    0.362
소매업     0.415
외식업     0.411
Name: _p_high, dtype: float64

비교:
      KOSIS 5년 생존율  우리 모델 평균 success
외식업           0.22             0.411
소매업           0.31             0.415
서비스업          0.47             0.362

순위 일치 — 서비스업 > 소매업 > 외식업: False


## §17.4 도메인 sanity — 실제 알려진 상권 직관 비교

§14.4 에서 6/6 중 일부 직관 불일치를 발견. 더 다양한 케이스로 보완 검증.

In [5]:
# 학습 + Quantile 모델 (행정동)
from sklearn.ensemble import HistGradientBoostingRegressor

df_ad = pd.read_csv(PROC / 'features_adstrd.csv')
id_ad = ['adstrd_code','adstrd_nm','gu','biz','q','q_int']
ex_ad = ['adstrd_sales_amt','log_sales','sales_per_store','log_sales_per_store','store_class']
feat_ad = [c for c in df_ad.columns if c not in id_ad + ex_ad]
for c in feat_ad: df_ad[c] = pd.to_numeric(df_ad[c], errors='coerce')
df_ad[feat_ad] = df_ad[feat_ad].fillna(df_ad[feat_ad].mean(numeric_only=True))

qm = {q: HistGradientBoostingRegressor(loss='quantile', quantile=q, max_depth=8,
                                       max_iter=400, learning_rate=0.1, random_state=42)
                    .fit(df_ad[feat_ad].values, df_ad['log_sales_per_store'].values)
      for q in [0.10, 0.50, 0.90]}

# 도메인 알려진 상권 케이스 (더 확장)
cases = [
    # (자치구, 행정동, 업종, 도메인 직관, 예상 등급)
    ('강남구','역삼1동','커피-음료',  '강남역 노른자',     'A'),
    ('서초구','서초2동','한식음식점','강남대로',         'A'),
    ('마포구','서교동',  '커피-음료',  '홍대 카페',        'A'),
    ('마포구','연남동',  '한식음식점','연트럴파크',       'A'),
    ('마포구','상수동',  '호프-간이주점','홍대 펍',       'A'),
    ('중구',  '명동',    '일반의류',    '명동 쇼핑',       'A'),
    ('중구',  '을지로동','한식음식점','을지로 노포',     'B'),
    ('종로구','종로1·2·3·4가동','한식음식점','광화문',  'A'),
    ('성동구','성수2가3동','커피-음료','성수카페거리',   'A'),
    ('용산구','이태원1동','일식음식점','이태원',         'B'),
    ('영등포구','여의동','커피-음료',  '여의도 오피스',   'A'),
    ('송파구','잠실3동','일반의류',    '잠실 롯데',       'A'),
    ('강북구','수유1동','한식음식점',  '동네 한식',        'C'),
    ('도봉구','쌍문2동','커피-음료',  '동네 카페',        'D'),
    ('관악구','신림동',  '한식음식점','신림 학사촌',     'B'),
    ('동대문구','용신동','일반의류',  '동대문 의류',      'A'),
]

def quick(gu, ad, biz):
    sub = df_ad[(df_ad['gu']==gu) & (df_ad['adstrd_nm']==ad) & (df_ad['biz']==biz)]
    if sub.empty:
        sub = df_ad[(df_ad['gu']==gu) & (df_ad['biz']==biz)]
        if sub.empty: return None, None, None, '데이터 없음'
        ad = '(자치구 평균)'
    row = sub.sort_values('q_int').iloc[-1]
    x = row[feat_ad].values.reshape(1,-1)
    p50 = float(np.exp(qm[0.50].predict(x)[0]))/1e6
    p90 = float(np.exp(qm[0.90].predict(x)[0]))/1e6
    return p50, p90, ad, None

print(f'{"위치":18s} {"업종":10s} {"P50":>5s} {"P90":>5s}  직관')
print('═' * 75)
for gu, ad, biz, intuition, expected in cases:
    p50, p90, matched, err = quick(gu, ad, biz)
    if err:
        print(f'{gu+" "+ad:18s} {biz:10s} {"--":>5s} {"--":>5s}  {err}')
        continue
    print(f'{gu+" "+matched:18s} {biz:10s} {p50:>4.0f}M {p90:>4.0f}M  {intuition} (예상 {expected})')

위치                 업종           P50   P90  직관
═══════════════════════════════════════════════════════════════════════════
강남구 역삼1동           커피-음료       409M  454M  강남역 노른자 (예상 A)
서초구 (자치구 평균)       한식음식점       115M  112M  강남대로 (예상 A)
마포구 서교동            커피-음료        14M   27M  홍대 카페 (예상 A)
마포구 연남동            한식음식점        13M   15M  연트럴파크 (예상 A)
마포구 (자치구 평균)       호프-간이주점      69M   67M  홍대 펍 (예상 A)
중구 명동              일반의류         22M   79M  명동 쇼핑 (예상 A)
중구 을지로동            한식음식점       159M  165M  을지로 노포 (예상 B)
종로구 종로1·2·3·4가동    한식음식점       101M  146M  광화문 (예상 A)
성동구 (자치구 평균)       커피-음료         1M    1M  성수카페거리 (예상 A)
용산구 (자치구 평균)       일식음식점        54M   50M  이태원 (예상 B)
영등포구 여의동           커피-음료        87M  121M  여의도 오피스 (예상 A)
송파구 (자치구 평균)       일반의류          2M    2M  잠실 롯데 (예상 A)
강북구 수유1동           한식음식점        47M   53M  동네 한식 (예상 C)
도봉구 (자치구 평균)       커피-음료        41M   41M  동네 카페 (예상 D)
관악구 (자치구 평균)       한식음식점        59M   77M  신림 학사촌 (예상 B)


동대문구 용신동           일반의류          1M   14M  동대문 의류 (예상 A)


## §17.5 종합

| § | 외부 검증 | 실제 결과 |
|---|---|---|
| 17.1 | 자치구·연·업종군 폐업 상관 | **양의 상관 +0.48~+0.75 (4가지 업종군 모두)** — 약점 노출 |
| 17.2 | 국세청 데이터 | 향후 다운로드 필요 — 코드 골격 제시 |
| 17.3 | KOSIS 5년 생존율 | **순위 불일치** (서비스업 47% vs 우리 0.36) — ⚠️ KOSIS 수치 출처 미확인(표 ID/URL 없음), 검증 불가 |
| 17.4 | 도메인 sanity 16케이스 | 약 절반만 직관 부합 — 홍대·명동·성수 과소평가 |

### 외부 검증 결과 — 정직한 해석

**1. 자치구 폐업과 양의 상관 +0.48~+0.75**
- 모델 결함이 아니라 **자치구 단위 규모 효과** (점포 많음 → 매출·폐업 동시)
- 페널티 강화(0.30→0.60)에도 -0.014 변화 (한계)
- 본질적 해결: 점포당 폐업률·업종별 5년 생존율·사업자 단위 추적 데이터 필요

**2. KOSIS 5년 생존율 순위 불일치** — ⚠️ 출처 미확인 (KOSIS 표 ID/URL 없이 인용, 검증 불가)
- KOSIS: 서비스업 47% > 소매업 31% > 외식업 22% (← 출처 미확인 수치)
- 우리: 소매업 0.42 > 외식업 0.41 > 서비스업 0.36
- → 표면상 서비스업 과소평가로 보이나, KOSIS 수치 자체가 검증되지 않아 이 순위 비교는 결론으로 인용 불가
- KOSIS는 진짜 생존, 우리는 매출 잠재력 → 차원이 다름 (단, 위 수치는 검증 전제)

**3. 서울시 변화지표 API와 r≈0 (§16)**
- 두 모델이 다른 차원 측정 — 보완재
- 거의 무관 = 우리 모델의 독립적 기여 증명

**4. 도메인 sanity 절반 fail**
- 홍대 P50 14M, 명동 P50 22M, 성수 P50 1M
- 행정동 평균이 작은 점포에 끌리는 한계
- 본질적 해결: 점포 단위 학습 (소상공인 상가정보) 필요

### 다음 단계 (소상공인진흥공단 API 키 발급 후)
```python
# .env 에 SBIZ_API_KEY 추가 후 자동 비교 코드
import os
SBIZ_KEY = os.getenv('SBIZ_API_KEY', '')
if SBIZ_KEY:
    # sg.sbiz.or.kr API 호출 → 점수 형태 입지 평가 받음
    # 우리 success_rate vs 소상공인 점수 직접 비교
    pass
```

### 보고서 §6.5 (validation) — 정직 표현

> "본 모델은 4가지 외부 검증을 수행했고 모두 **약점·한계를 정직히 드러낸다**.
> - **자치구 폐업 양의 상관 +0.48~+0.75** — 규모 효과 (모델 결함 X, 자치구 단위 한계)
> - **KOSIS 5년 생존율 순위 불일치** — 매출 분위 vs 진짜 생존의 차원 차이 (⚠️ KOSIS 수치 출처 미확인, 검증 불가)
> - **서울시 API와 r≈0** — 다른 차원 측정 (강점이자 한계)
> - **도메인 sanity 절반 fail** — 행정동 평균의 한계 (점포 단위 데이터 필요)
>
> 본 모델은 '완벽한 평가기'가 아니라 **'공개 데이터로 가능한 매출 잠재력 1차 스크리닝 도구'** 임을 외부 검증으로 인정한다. 진짜 생존 평가는 후속 작업(국세청·소상공인 상가정보)으로 가능하다.